# 🤟 Wasel v4 Pro — Live Sign Language Translator
### Real-time PSL (Pakistan Sign Language) translation using YOLOv8-Pose + LSTM

**How to use:**
1. Run Cell 1 (Install) → Wait for ✅
2. Run Cell 2 (Setup) → Wait for ✅
3. Run Cell 3 (Launch) → Click the `*.gradio.live` link!

> ⚡ **No Docker. No Cloud Shell. Just run and demo!**

In [ ]:
#@title 📦 Cell 1: Install Dependencies (Run Once)
#@markdown Takes ~2 minutes. Do NOT downgrade numpy — Colab needs numpy>=2.0.

print("⏳ Installing dependencies...")
print("   (TensorFlow is pre-installed on Colab — skipping)")

# LESSONS LEARNED:
# 1. Do NOT install numpy<2.0 — Colab 3.12 has JAX/cupy that REQUIRE numpy>=2.0
# 2. Do NOT install tensorflow — Colab already has it pre-installed and compatible
# 3. Pin fastrtc>=0.0.34 — older versions don't have Stream class
# 4. Pin gradio>=5.0.0 — required by fastrtc>=0.0.34

!pip install -q \
    "gradio>=5.0.0" \
    "fastrtc>=0.0.34" \
    "ultralytics>=8.0.0" \
    "scikit-learn>=1.3.0"

print("\n🔍 Verifying imports...")

import importlib
all_ok = True
for name, mod in {"gradio": "gradio", "ultralytics": "ultralytics", "sklearn": "sklearn", "cv2": "cv2", "numpy": "numpy"}.items():
    try:
        m = importlib.import_module(mod)
        print(f"   ✅ {name}: {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"   ❌ {name}: {e}")
        all_ok = False

# Check TensorFlow separately (pre-installed)
try:
    import tensorflow as tf
    print(f"   ✅ tensorflow: {tf.__version__} (pre-installed)")
except Exception as e:
    print(f"   ⚠️ tensorflow: {e} — will use sklearn classifier instead")

# Critical check: fastrtc.Stream
try:
    from fastrtc import Stream
    print(f"   ✅ fastrtc.Stream: Available")
except ImportError as e:
    print(f"   ❌ fastrtc.Stream: {e}")
    all_ok = False

if all_ok:
    print("\n🎉 All dependencies ready! Proceed to Cell 2.")
else:
    print("\n⚠️ Issues detected. Try: Runtime → Restart Session, then re-run.")

In [ ]:
#@title 🧠 Cell 2: Setup AI Engine
#@markdown Downloads YOLO model + loads the trained PSL classifier.

import os, cv2, pickle, logging
import numpy as np
from collections import deque

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("wasel")

# ─── Clone repo for the trained model ───
REPO_DIR = "/content/wasel_repo"
if not os.path.exists(REPO_DIR):
    print("⏳ Cloning repository...")
    !git clone -q https://github.com/eltaweelactuary/Wasel_v4_Official_Release.git {REPO_DIR}
    print("   ✅ Repository cloned.")
else:
    print("   ✅ Repository already exists.")

# ─── Load YOLO ───
print("⏳ Loading YOLOv8-Pose...")
from ultralytics import YOLO
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo_model = YOLO("yolov8n-pose.pt").to(device)
print(f"   ✅ YOLOv8n-Pose loaded on {device}.")

# ─── Load Classifier ───
classifier, label_encoder = None, None
PKL_PATH = os.path.join(REPO_DIR, "GCP_Source_Code/wasel_v4_data/models/psl_classifier.pkl")

if os.path.exists(PKL_PATH) and os.path.getsize(PKL_PATH) > 100:
    with open(PKL_PATH, 'rb') as f:
        classifier, label_encoder = pickle.load(f)
    print(f"   ✅ PSL Classifier loaded ({os.path.getsize(PKL_PATH)//1024} KB)")
else:
    print(f"   ⚠️ Classifier not found — predictions will show 'No model'.")

# ─── Frame Processing ───
sequence_buffer = deque(maxlen=30)

def process_frame(image):
    """Process a single WebRTC frame: YOLO skeleton + sign prediction."""
    if image is None:
        return None
    try:
        # Faster image resizing (optional, YOLO handles this anyway, but this is safe)
        # YOLO Pose detection
        results = yolo_model(image, verbose=False)
        annotated = results[0].plot(boxes=False)

        # Extract keypoints
        label_text = "Waiting for signs..."
        if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
            kp = results[0].keypoints.data[0].cpu().numpy().flatten()
            sequence_buffer.append(kp)

            if classifier and len(sequence_buffer) > 5:
                features = np.mean(np.array(list(sequence_buffer)), axis=0).reshape(1, -1)
                expected = classifier.n_features_in_
                actual = features.shape[1]
                if actual > expected:
                    features = features[:, :expected]
                elif actual < expected:
                    features = np.pad(features, ((0, 0), (0, expected - actual)))
                probs = classifier.predict_proba(features)[0]
                idx = np.argmax(probs)
                conf = float(probs[idx]) * 100
                if conf > 35.0:
                    label = label_encoder.inverse_transform([idx])[0]
                    label_text = f"{label} ({conf:.1f}%)"

        # Draw prediction text
        cv2.putText(annotated, label_text, (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        # Motion energy
        if len(sequence_buffer) >= 2:
            seq = np.array(list(sequence_buffer))
            n_cols = min(seq.shape[1], 126)
            energy = np.linalg.norm(np.diff(seq[:, :n_cols], axis=0))
            if energy > 2.0:
                cv2.putText(annotated, "Signing Detected", (30, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        return annotated
    except Exception as e:
        logger.error(f"Frame error: {e}")
        return image

print("\n🚀 AI Engine ready! Proceed to Cell 3.")

In [ ]:
#@title 🎥 Cell 3: Launch Live Demo (Requires HF Token)
#@markdown Colab blocks WebRTC UDP traffic. We must use a TURN server.
#@markdown Get a **free** token from: [Hugging Face Tokens](https://huggingface.co/settings/tokens) 

HF_TOKEN = "" #@param {type:"string"}

import os
from fastrtc import Stream, get_cloudflare_turn_credentials

if not HF_TOKEN:
    raise ValueError("❌ You MUST paste a Hugging Face token in the box above before running this cell!")

os.environ["HF_TOKEN"] = HF_TOKEN.strip()

print("⏳ Getting free Cloudflare TURN server credentials...")
rtc_config = get_cloudflare_turn_credentials()

stream = Stream(
    handler=process_frame,
    modality="video",
    mode="send-receive",
    rtc_configuration=rtc_config
)

print("\n🎉 Launching Wasel v4 Live Demo...")
print("📋 A public URL will appear below — share it with the client!")
print("⚠️  Keep this cell running. Closing it stops the demo.\n")

stream.ui.launch(share=True, quiet=False)